# 프로토콜 v1.1 — A1·A2: 기존 시험 예측으로 하는 추가 분석 (탐색 분석)

**새 학습이나 새 예측은 없습니다.** Phase 3에서 고정된 시험 예측만 다시 계산합니다.

- **A1:** 예측을 물리적 최대치(RUL ≤ 100 − t)로 자른 RMSE로 v1.0과 같은 계층 붓스트랩 검정을 합니다 (V0 대 RF, GB, MLP, LSTM, CNN). 보정하지 않은 p값과 Holm 보정 p값을 둘 다 냅니다.
- **A2:** 보정한 90% 구간(고정된 c₉₀)의 서브셋별·엔진별 포함률, 최악의 엔진 포함률, LB90 한쪽 보정(실제 RUL < LB90인 비율, 기대값 5%)을 서브셋별·수명 구간별로 냅니다.

**실행 전:** 첫 셀의 `ADDENDUM_COMMIT_URL`에 v1.1 커밋 주소를 넣으세요. 비어 있으면 실행되지 않습니다.
**결과:** 드라이브 `addendum_v11/` (`a1_clipped_tests.csv`, `a2_coverage_*.csv`, `a12_summary.txt`). 런타임은 CPU면 됩니다.

In [ ]:
import os, sys, json, glob, time, hashlib, re
import numpy as np, pandas as pd
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/PI-BDP_NCMAPSS'
else:
    ROOT = os.environ.get('NCM_ROOT', './ncm')
IN3 = f'{ROOT}/phase3'
OUT = f'{ROOT}/addendum_v11'; os.makedirs(OUT, exist_ok=True)
cfg = json.load(open(f'{ROOT}/phase1/config.json')); SUBSETS = sorted(cfg['k'])
K = ['subset', 'unit', 'cycle']

# 프로토콜 v1.1을 커밋한 주소 (실행 전에 커밋해야 합니다)
ADDENDUM_COMMIT_URL = ''
assert re.match(r'https://github\.com/.+/commit/[0-9a-f]{7,40}$', ADDENDUM_COMMIT_URL), \
    'v1.1 커밋 주소를 먼저 넣으세요. 추가본을 커밋하기 전에는 실행하지 않습니다.'
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''): h.update(b)
    return h.hexdigest()
# Phase 3 시험 파일이 고정 당시와 같은지 확인
REF = json.load(open(f'{IN3}/sha256_phase3.json'))
bad = [k for k, v in REF.items() if os.path.exists(f'{IN3}/{k}') and sha(f'{IN3}/{k}') != v]
assert not bad, f'Phase 3 결과 파일이 바뀌었습니다: {bad}'
print('ROOT', ROOT, '| 서브셋', SUBSETS, '| Phase 3 해시 확인', len(REF), '개')

In [ ]:
# === v1.0 Phase 3와 같은 함수 (pibdp_ncm.py에서 그대로 옮김) ===
CAP = 100
def clip_physical(pred, cycle, cap=CAP): return np.clip(pred, 0, cap - np.asarray(cycle))
def hier_boot_rmse_diff(A, Bm, n_boot=100000, seed=0):
    M = A[K + ['pred', 'true']].merge(Bm[K + ['pred']], on=K, suffixes=('_a', '_b'))
    assert len(M) == len(A) == len(Bm), 'rows differ'
    M['ea'] = (M.pred_a - M.true) ** 2; M['eb'] = (M.pred_b - M.true) ** 2
    U = M.groupby(['subset', 'unit']).agg(sa=('ea', 'sum'), sb=('eb', 'sum'), n=('ea', 'size')).reset_index()
    subsets = sorted(U.subset.unique()); rng = np.random.default_rng(seed)
    per = [U[U.subset == s][['sa', 'sb', 'n']].values for s in subsets]
    SA = np.zeros(n_boot); SB = np.zeros(n_boot); N = np.zeros(n_boot)
    pick = rng.integers(0, len(subsets), (n_boot, len(subsets)))
    for j in range(len(subsets)):
        for si, arr in enumerate(per):
            rows = np.flatnonzero(pick[:, j] == si)
            if not len(rows): continue
            idx = rng.integers(0, len(arr), (len(rows), len(arr)))
            tot = arr[idx].sum(1)
            SA[rows] += tot[:, 0]; SB[rows] += tot[:, 1]; N[rows] += tot[:, 2]
    d = np.sqrt(SA / N) - np.sqrt(SB / N)
    obs = float(np.sqrt(U.sa.sum() / U.n.sum()) - np.sqrt(U.sb.sum() / U.n.sum()))
    p = float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))
    return dict(diff=obs, ci_lo=float(np.percentile(d, 2.5)), ci_hi=float(np.percentile(d, 97.5)), p_boot=p)
def holm(pvals):
    p = np.asarray(pvals, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for r, i in enumerate(o):
        run = max(run, min(1.0, (m - r) * p[i])); adj[i] = run
    return adj
def rmse(e): return float(np.sqrt(np.mean(np.square(e))))

In [ ]:
# === 고정된 시험 예측 불러오기 (Phase 3와 같은 방식) ===
V0 = pd.concat([pd.read_csv(f'{IN3}/test_PI-BDP_V0_{s}.csv.gz') for s in SUBSETS], ignore_index=True)
V0 = V0.rename(columns={'q500': 'pred'}).sort_values(K).reset_index(drop=True)
COMP = ['RF', 'GB', 'MLP', 'LSTM', 'CNN']
ENS = {n: pd.read_csv(f'{IN3}/test_{n}.csv.gz').groupby(K, as_index=False)[['pred', 'true']].mean().sort_values(K).reset_index(drop=True)
       for n in COMP}
for n, df in ENS.items():
    assert (df[K].values == V0[K].values).all(), f'{n}: 채점 시점이 다름'
print('채점 시점', len(V0), '| 엔진', V0.groupby(['subset', 'unit']).ngroups)

In [ ]:
# === A1: 물리적 최대치로 자른 예측의 계층 붓스트랩 검정 ===
def clipped(df): return df.assign(pred=clip_physical(df.pred.values, df.cycle.values))
V0c = clipped(V0); rows = []
for n in COMP:
    r = hier_boot_rmse_diff(V0c, clipped(ENS[n]), 100000, seed=1)
    rows.append(dict(comparison=f'V0 - {n} (clipped)', rmse_v0=rmse(V0c.pred - V0c.true),
                     rmse_comp=rmse(clipped(ENS[n]).pred - ENS[n].true), **r))
A1 = pd.DataFrame(rows); A1['p_holm'] = holm(A1.p_boot.values)
A1.to_csv(f'{OUT}/a1_clipped_tests.csv', index=False)
print(A1.round(4).to_string(index=False))

In [ ]:
# === A2: 보정한 90% 구간의 포함률과 LB90 한쪽 보정 ===
c90 = json.load(open(f'{ROOT}/phase1/calibration.json'))['c']['90']; EPS = 0.5
m, y = V0.pred.values, V0.true.values
L = np.maximum(m - c90 * np.maximum(m - V0.q050.values, EPS), 0); U = m + c90 * np.maximum(V0.q950.values - m, EPS)
D = V0[K + ['true']].assign(L=L, U=U, inside=((y >= L) & (y <= U)).astype(float), below=(y < L).astype(float),
                            below_raw=(y < V0.q050.values).astype(float),
                            iscore=(U - L) + 20 * (L - y) * (y < L) + 20 * (y - U) * (y > U))
f = D.cycle / (D.cycle + D.true)
D['life_bin'] = pd.cut(f, [0, .25, .5, .75, 1.0001], labels=['0-25%', '25-50%', '50-75%', '75-100%'], include_lowest=True)
agg = dict(n=('inside', 'size'), cov90=('inside', 'mean'), below_LB90=('below', 'mean'), below_q05_raw=('below_raw', 'mean'),
           iscore90=('iscore', 'mean'))
by_subset = D.groupby('subset').agg(**agg)
by_engine = D.groupby(['subset', 'unit']).agg(**agg).reset_index()
by_bin = D.groupby('life_bin', observed=True).agg(**agg)
by_subset_bin = D.groupby(['subset', 'life_bin'], observed=True).agg(**agg).reset_index()
overall = D.agg({'inside': 'mean', 'below': 'mean', 'below_raw': 'mean', 'iscore': 'mean'})
worst = by_engine.sort_values('cov90').head(5)
for nm, t in [('subset', by_subset), ('engine', by_engine), ('lifebin', by_bin), ('subset_lifebin', by_subset_bin)]:
    t.to_csv(f'{OUT}/a2_coverage_{nm}.csv')
print('전체: cov90 %.3f | 실제 RUL < LB90 %.3f (기대 0.05) | 보정 전 q05 아래 %.3f' % (overall.inside, overall.below, overall.below_raw))
print(by_subset.round(3).to_string()); print('\n가장 낮은 엔진 5대:\n', worst.round(3).to_string(index=False))
print('\n수명 구간별:\n', by_bin.round(3).to_string())

In [ ]:
# === 요약 저장 + 해시 ===
txt = '\n'.join([f"Addendum v1.1 A1/A2 (exploratory) {time.strftime('%Y-%m-%d %H:%M:%S')}  addendum={ADDENDUM_COMMIT_URL}",
    '', 'A1: clipped predictions (RUL <= 100 - t), paired hierarchical bootstrap, 100,000 resamples:', A1.round(4).to_string(index=False),
    '', f'A2: calibrated 90% interval (c90 = {c90:.4f}); overall coverage {overall.inside:.4f}, '
        f'fraction true RUL < LB90 {overall.below:.4f} (nominal 0.05), raw q05 {overall.below_raw:.4f}',
    '', 'By subset:', by_subset.round(4).to_string(), '', 'Lowest-coverage engines:', worst.round(4).to_string(index=False),
    '', 'By life bin:', by_bin.round(4).to_string()])
open(f'{OUT}/a12_summary.txt', 'w').write(txt)
H = {os.path.basename(p): sha(p) for p in sorted(glob.glob(f'{OUT}/a1_*') + glob.glob(f'{OUT}/a2_*') + [f'{OUT}/a12_summary.txt'])}
json.dump(H, open(f'{OUT}/sha256_a12.json', 'w'), indent=1); print(txt)